In [12]:
import ee

GEE_PROJECT = "ee-josaiahsyiem"

# authenticate (opens a browser the first time; remembered after)
try:
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine ready. Project:", GEE_PROJECT)
except Exception as e:
    print("Need to authenticate first. Running authentication...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine ready. Project:", GEE_PROJECT)

Earth Engine ready. Project: ee-josaiahsyiem


In [13]:
from pathlib import Path
import geopandas as gpd

# load fields
FIELDS = Path(r"F:\THESIS\mtech_thesis_crop_yield_estimation\data\processed\yield_analysis_ready.gpkg")
gdf = gpd.read_file(FIELDS)
print("fields loaded:", len(gdf))

# pick ONE test field from 2024
test = gdf[gdf["year"] == 2024].iloc[0]
print("test field:", test["uid"], "| village:", test["village"], "| year:", test["year"])

# convert its polygon to an Earth Engine geometry
geom = test.geometry
ee_geom = ee.Geometry(geom.__geo_interface__)

# Sentinel-1 collection for this field's 2024 kharif season (June to Nov)
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(ee_geom)
      .filterDate("2024-06-01", "2024-11-30")
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

print("\nSentinel-1 scenes found for this field's 2024 season:", s1.size().getInfo())

# how many per orbit direction?
asc = s1.filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
desc = s1.filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
print("  ascending scenes:", asc.size().getInfo())
print("  descending scenes:", desc.size().getInfo())

fields loaded: 1204
test field: 6830-7688-5916 | village: Murud Akola | year: 2024

Sentinel-1 scenes found for this field's 2024 season: 14
  ascending scenes: 0
  descending scenes: 14


In [14]:
# descending only (the only orbit available here)
s1_desc = s1.filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))

# negative buffer ~10m to avoid mixed edge pixels (0.0001 deg ~ 11m)
field_core = ee_geom.buffer(-10)

def extract_scene(img):
    # S1_GRD is already in dB. Get mean VV and VH inside the field core.
    stats = img.select(["VV", "VH"]).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=field_core,
        scale=10,
        maxPixels=1e9,
    )
    return ee.Feature(None, {
        "date": img.date().format("YYYY-MM-dd"),
        "VV": stats.get("VV"),
        "VH": stats.get("VH"),
    })

# run over all 14 scenes and pull the table
feats = s1_desc.map(extract_scene)
data = feats.getInfo()["features"]

import pandas as pd
rows = [{"date": f["properties"]["date"],
         "VV": f["properties"].get("VV"),
         "VH": f["properties"].get("VH")} for f in feats.getInfo()["features"]]
ts = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
ts["VH_VV_ratio"] = ts["VH"] - ts["VV"]  # in dB, ratio = difference

print("SAR time series for test field", test["uid"], "(2024 season):")
print(ts.to_string(index=False))
print()
print("VV range:", round(ts['VV'].min(),2), "to", round(ts['VV'].max(),2), "dB")
print("VH range:", round(ts['VH'].min(),2), "to", round(ts['VH'].max(),2), "dB")
print("any missing:", ts[['VV','VH']].isna().any().any())

SAR time series for test field 6830-7688-5916 (2024 season):
      date         VV         VH  VH_VV_ratio
2024-06-07  -7.405546 -19.186128   -11.780582
2024-06-19 -12.682657 -25.748901   -13.066243
2024-07-01  -9.598131 -19.488713    -9.890582
2024-07-13  -9.284315 -19.261505    -9.977191
2024-07-25  -7.093621 -13.953117    -6.859496
2024-08-06  -7.801853 -15.027247    -7.225394
2024-08-18  -5.795589 -15.710754    -9.915164
2024-08-30  -7.349212 -14.996799    -7.647587
2024-09-11  -7.973678 -15.679307    -7.705629
2024-09-23  -9.532909 -17.709811    -8.176903
2024-10-05 -10.514135 -15.737421    -5.223286
2024-10-29  -9.458799 -18.746579    -9.287780
2024-11-10  -9.421371 -20.263632   -10.842261
2024-11-22  -8.872564 -19.963159   -11.090595

VV range: -12.68 to -5.8 dB
VH range: -25.75 to -13.95 dB
any missing: False


In [15]:
import pandas as pd
import numpy as np

# define the 12 fortnight windows (month, half) for a June-Nov season
FORTNIGHTS = []
for month in [6, 7, 8, 9, 10, 11]:
    FORTNIGHTS.append((month, 1))  # 1st half (day 1-15)
    FORTNIGHTS.append((month, 2))  # 2nd half (day 16-end)

FN_NAMES = []
month_abbr = {6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov"}
for m, half in FORTNIGHTS:
    FN_NAMES.append(f"{half}FN_{month_abbr[m]}")

def assign_fortnight(date_str):
    d = pd.to_datetime(date_str)
    half = 1 if d.day <= 15 else 2
    return (d.month, half)

# take the raw time series 'ts' from Cell 3 and bin into 12 fortnights
ts["fn"] = ts["date"].apply(assign_fortnight)

fn_vv, fn_vh = [], []
for fn in FORTNIGHTS:
    scenes = ts[ts["fn"] == fn]
    if len(scenes):
        # average in LINEAR power then back to dB (proper way)
        vv_lin = (10 ** (scenes["VV"] / 10)).mean()
        vh_lin = (10 ** (scenes["VH"] / 10)).mean()
        fn_vv.append(10 * np.log10(vv_lin))
        fn_vh.append(10 * np.log10(vh_lin))
    else:
        fn_vv.append(np.nan)
        fn_vh.append(np.nan)

fortnight_df = pd.DataFrame({
    "fortnight": FN_NAMES,
    "VV": np.round(fn_vv, 2),
    "VH": np.round(fn_vh, 2),
})
fortnight_df["VH_VV_ratio"] = (fortnight_df["VH"] - fortnight_df["VV"]).round(2)

print("12-fortnight SAR for test field", test["uid"], "(2024):")
print(fortnight_df.to_string(index=False))
print()
print("filled fortnights:", fortnight_df["VV"].notna().sum(), "of 12")

12-fortnight SAR for test field 6830-7688-5916 (2024):
fortnight     VV     VH  VH_VV_ratio
  1FN_Jun  -7.41 -19.19       -11.78
  2FN_Jun -12.68 -25.75       -13.07
  1FN_Jul  -9.44 -19.37        -9.93
  2FN_Jul  -7.09 -13.95        -6.86
  1FN_Aug  -7.80 -15.03        -7.23
  2FN_Aug  -6.50 -15.34        -8.84
  1FN_Sep  -7.97 -15.68        -7.71
  2FN_Sep  -9.53 -17.71        -8.18
  1FN_Oct -10.51 -15.74        -5.23
  2FN_Oct  -9.46 -18.75        -9.29
  1FN_Nov  -9.42 -20.26       -10.84
  2FN_Nov  -8.87 -19.96       -11.09

filled fortnights: 12 of 12


In [20]:
import pandas as pd
import numpy as np

month_abbr = {6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov"}
MONTHS = [6, 7, 8, 9, 10, 11]
MONTH_NAMES = [month_abbr[m] for m in MONTHS]

def month_dates(year, month):
    nm = month + 1 if month < 12 else 1
    ny = year if month < 12 else year + 1
    return f"{year}-{month:02d}-01", f"{ny}-{nm:02d}-01"

# ---- simple speckle filter: focal mean on linear power ----
def speckle_filter(img_linear):
    return img_linear.focal_mean(radius=30, units="meters")

def extract_year_sar(gdf_year, year, buffer_m=-10):
    feats, skipped = [], 0
    for _, row in gdf_year.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            skipped += 1
            continue
        g = ee.Geometry(row.geometry.__geo_interface__).buffer(buffer_m)
        feats.append(ee.Feature(g, {"uid": row["uid"]}))
    if skipped:
        print(f"[skipped {skipped} no-geom]", end=" ")
    fc = ee.FeatureCollection(feats)

    base = (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
            .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING")))

    month_images = []
    for m, name in zip(MONTHS, MONTH_NAMES):
        d0, d1 = month_dates(year, m)
        coll = base.filterDate(d0, d1).select(["VV", "VH"])
        def to_lin_filtered(img):
            lin = ee.Image(10).pow(img.divide(10))
            return speckle_filter(lin)
        filt = coll.map(to_lin_filtered)
        med = filt.median()
        db = med.log10().multiply(10).rename([f"VV_{name}", f"VH_{name}"])
        month_images.append(db)

    stack = ee.Image.cat(month_images)
    result = stack.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=10)
    rows = result.getInfo()["features"]

    recs = []
    for r in rows:
        p = r["properties"]
        rec = {"uid": p["uid"]}
        for name in MONTH_NAMES:
            rec[f"VV_{name}"] = p.get(f"VV_{name}")
            rec[f"VH_{name}"] = p.get(f"VH_{name}")
        recs.append(rec)
    return pd.DataFrame(recs)

In [21]:
import pandas as pd
import numpy as np
import time

all_sar = []
for yr in [2022, 2023, 2024, 2025]:
    gdf_yr = gdf[gdf["year"] == yr].copy()
    print(f"extracting {yr}: {len(gdf_yr)} fields...", end=" ")
    t0 = time.time()
    sar_yr = extract_year_sar(gdf_yr, yr)
    sar_yr["year"] = yr
    all_sar.append(sar_yr)
    print(f"done in {round(time.time()-t0,1)}s | shape {sar_yr.shape}")

# combine all years
sar_all = pd.concat(all_sar, ignore_index=True)

# add VH/VV ratio for each month (in dB, ratio = VH - VV)
for name in MONTH_NAMES:
    sar_all[f"ratio_{name}"] = sar_all[f"VH_{name}"] - sar_all[f"VV_{name}"]

print("\ncombined SAR table:", sar_all.shape)
print("fields per year:")
print(sar_all["year"].value_counts().sort_index().to_string())
print()

# how complete is the SAR data? count fully-filled fields
sar_cols = [c for c in sar_all.columns if c.startswith(("VV_", "VH_"))]
sar_all["sar_complete"] = sar_all[sar_cols].notna().all(axis=1)
print("fields with ALL 24 SAR values filled:", int(sar_all["sar_complete"].sum()), "of", len(sar_all))
print("fields with at least SOME SAR:", int(sar_all[sar_cols].notna().any(axis=1).sum()))

extracting 2022: 55 fields... done in 6.4s | shape (55, 14)
extracting 2023: 58 fields... done in 6.3s | shape (58, 14)
extracting 2024: 323 fields... done in 18.3s | shape (323, 14)
extracting 2025: 768 fields... [skipped 1 no-geom] done in 33.9s | shape (767, 14)

combined SAR table: (1203, 20)
fields per year:
year
2022     55
2023     58
2024    323
2025    767

fields with ALL 24 SAR values filled: 1197 of 1203
fields with at least SOME SAR: 1197


In [22]:
from pathlib import Path

PROCESSED = Path(r"F:\THESIS\mtech_thesis_crop_yield_estimation\data\processed")

# drop the helper column and save
sar_out = sar_all.drop(columns=["sar_complete"], errors="ignore")
out = PROCESSED / "sar_monthly_features.csv"
sar_out.to_csv(out, index=False)

print("saved:", out)
print("shape:", sar_out.shape)
print("columns:", list(sar_out.columns))

saved: F:\THESIS\mtech_thesis_crop_yield_estimation\data\processed\sar_monthly_features.csv
shape: (1203, 20)
columns: ['uid', 'VV_Jun', 'VH_Jun', 'VV_Jul', 'VH_Jul', 'VV_Aug', 'VH_Aug', 'VV_Sep', 'VH_Sep', 'VV_Oct', 'VH_Oct', 'VV_Nov', 'VH_Nov', 'year', 'ratio_Jun', 'ratio_Jul', 'ratio_Aug', 'ratio_Sep', 'ratio_Oct', 'ratio_Nov']
